## 1. Load Model-Ready Data

Load the feature matrices and target arrays generated in Notebook 02. Verify their dimensions, data types, and target values before training regression models.

In [1]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

feature_path = Path("../data/model_features.joblib")

if not feature_path.exists():
    raise FileNotFoundError(f"Feature file not found: {feature_path.resolve()}")

data = joblib.load(feature_path)

X_train = data["X_train"]
X_valid = data["X_valid"]
y_train = data["y_train"]
y_valid = data["y_valid"]

print("Training features:", X_train.shape)
print("Validation features:", X_valid.shape)
print("Training targets:", y_train.shape)
print("Validation targets:", y_valid.shape)

print("\nTraining target summary:")
print(pd.Series(y_train).describe())

print("\nData types:")
print("X_train:", X_train.dtypes.value_counts().to_dict())
print("Target:", y_train.dtype)

Training features: (1185328, 212)
Validation features: (296333, 212)
Training targets: (1185328,)
Validation targets: (296333,)

Training target summary:
count    1.185328e+06
mean     2.976707e+00
std      7.311020e-01
min      1.386294e+00
25%      2.397895e+00
50%      2.890372e+00
75%      3.401197e+00
max      5.141664e+00
dtype: float64

Data types:
X_train: {dtype('float32'): 200, dtype('int64'): 8, dtype('float64'): 4}
Target: float64


## 2. Baseline Model

A baseline establishes a minimum performance reference before testing more sophisticated algorithms. Here, a `DummyRegressor` predicts the mean log-price from the training data for every validation listing.

The model is evaluated using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). Both metrics are calculated in log-price space, so they are not directly interpretable as errors in the original currency.

In [2]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_valid)

baseline_mae = mean_absolute_error(y_valid, baseline_predictions)
baseline_rmse = np.sqrt(
    mean_squared_error(y_valid, baseline_predictions)
)

print(f"Baseline MAE (log-price):  {baseline_mae:.4f}")
print(f"Baseline RMSE (log-price): {baseline_rmse:.4f}")

Baseline MAE (log-price):  0.5817
Baseline RMSE (log-price): 0.7455


## 3. Ridge Regression

Ridge Regression is a linear regression model that applies L2 regularization to reduce the effect of large coefficients. It provides a useful first machine-learning benchmark for the structured and reduced-dimensional text features.

The model is trained on the training dataset and evaluated on the held-out validation dataset using MAE and RMSE in log-price space. The validation dataset is used for evaluation, not model fitting.

In [3]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

ridge_model = Ridge(alpha=1.0, solver="lsqr")

ridge_model.fit(X_train, y_train)

ridge_predictions = ridge_model.predict(X_valid)

ridge_mae = mean_absolute_error(y_valid, ridge_predictions)
ridge_rmse = np.sqrt(
    mean_squared_error(y_valid, ridge_predictions)
)

print(f"Ridge MAE (log-price):  {ridge_mae:.4f}")
print(f"Ridge RMSE (log-price): {ridge_rmse:.4f}")

Ridge MAE (log-price):  0.5643
Ridge RMSE (log-price): 0.7219


## 4. Tree-Based Regression — Random Forest

Random Forest is an ensemble learning algorithm that trains multiple decision trees on different samples of the training data and averages their predictions for regression.

Because the dataset contains over one million training records and the system has 16 GB of RAM, an initial experiment will use a subset of the training data. This allows us to evaluate feasibility before considering larger training runs.

The model will be evaluated using MAE and RMSE in log-price space.

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Use a reproducible subset for the initial experiment
sample_size = min(100_000, len(X_train))

X_sample = X_train.sample(
    n=sample_size,
    random_state=42
)
y_sample = y_train[X_sample.index]

rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=5,
    max_features=0.7,
    n_jobs=4,
    random_state=42
)

rf_model.fit(X_sample, y_sample)

rf_predictions = rf_model.predict(X_valid)

rf_mae = mean_absolute_error(y_valid, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_valid, rf_predictions))

print(f"Random Forest MAE (log-price):  {rf_mae:.4f}")
print(f"Random Forest RMSE (log-price): {rf_rmse:.4f}")

Random Forest MAE (log-price):  0.4535
Random Forest RMSE (log-price): 0.5937


In [6]:
import importlib.util

print("LightGBM installed:", importlib.util.find_spec("lightgbm") is not None)

LightGBM installed: True


## 5. Boosting Regression — LightGBM

LightGBM is a gradient-boosting framework that builds decision trees sequentially, with each new tree improving the model's predictions. It is designed for efficient training on large tabular datasets.

To make the initial experiment practical on a system with 16 GB RAM, the model is trained on the same 100,000-row sample used for the Random Forest experiment. Performance is evaluated on the full validation dataset using MAE and RMSE in log-price space.

In [7]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import time

# Reuse X_sample and y_sample from the Random Forest experiment
lgbm_model = LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    max_bin=127,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=4,
    random_state=42,
    verbosity=-1
)

start_time = time.time()

lgbm_model.fit(X_sample, y_sample)

training_time = time.time() - start_time

lgbm_predictions = lgbm_model.predict(X_valid)

lgbm_mae = mean_absolute_error(y_valid, lgbm_predictions)
lgbm_rmse = np.sqrt(
    mean_squared_error(y_valid, lgbm_predictions)
)

print(f"LightGBM MAE (log-price):  {lgbm_mae:.4f}")
print(f"LightGBM RMSE (log-price): {lgbm_rmse:.4f}")
print(f"Training time: {training_time:.1f} seconds")

LightGBM MAE (log-price):  0.4206
LightGBM RMSE (log-price): 0.5519
Training time: 15.4 seconds


## 6. Hyperparameter Tuning — LightGBM

Hyperparameter tuning searches for model configurations that improve predictive performance. `RandomizedSearchCV` evaluates a limited number of randomly selected configurations using three-fold cross-validation on the training sample.

The search optimizes negative MAE, equivalent to minimizing MAE. The selected configuration is then evaluated on the separate validation dataset to compare it with the initial LightGBM model.

In [8]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import randint, uniform
import numpy as np
import time

base_lgbm = LGBMRegressor(
    objective="regression",
    max_bin=127,
    n_jobs=4,
    random_state=42,
    verbosity=-1
)

param_distributions = {
    "n_estimators": randint(200, 601),
    "learning_rate": uniform(0.02, 0.08),
    "num_leaves": randint(20, 61),
    "max_depth": [-1, 8, 12, 16],
    "min_child_samples": randint(20, 101),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_lambda": uniform(0.0, 5.0),
}

search = RandomizedSearchCV(
    estimator=base_lgbm,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="neg_mean_absolute_error",
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=1,
    refit=True
)

start_time = time.time()
search.fit(X_sample, y_sample)
search_time = time.time() - start_time

tuned_lgbm = search.best_estimator_
tuned_predictions = tuned_lgbm.predict(X_valid)

tuned_mae = mean_absolute_error(y_valid, tuned_predictions)
tuned_rmse = np.sqrt(
    mean_squared_error(y_valid, tuned_predictions)
)

print("\nBest parameters:")
print(search.best_params_)

print(f"\nBest CV MAE: {-search.best_score_:.4f}")
print(f"Tuned validation MAE:  {tuned_mae:.4f}")
print(f"Tuned validation RMSE: {tuned_rmse:.4f}")
print(f"Search time: {search_time / 60:.1f} minutes")

Fitting 3 folds for each of 6 candidates, totalling 18 fits

Best parameters:
{'colsample_bytree': np.float64(0.9165996316800473), 'learning_rate': np.float64(0.09508421672126002), 'max_depth': 8, 'min_child_samples': 83, 'n_estimators': 476, 'num_leaves': 52, 'reg_lambda': np.float64(1.5212112147976886)}

Best CV MAE: 0.4139
Tuned validation MAE:  0.4109
Tuned validation RMSE: 0.5411
Search time: 3.5 minutes


## 7. Save the Tuned Model

Save the best LightGBM estimator and its selected hyperparameters so that the model can be reused without repeating the hyperparameter search.

The saved estimator was trained on the 100,000-row training sample used during tuning.

In [9]:
import joblib
from pathlib import Path

model_dir = Path("../data/models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "model": tuned_lgbm,
        "best_params": search.best_params_,
        "validation_mae": tuned_mae,
        "validation_rmse": tuned_rmse,
    },
    model_dir / "tuned_lightgbm.joblib",
    compress=3
)

print("Tuned LightGBM saved successfully.")

Tuned LightGBM saved successfully.


## 8. Voting Ensemble

A voting regressor combines predictions from multiple regression models by averaging their outputs. This experiment combines Ridge Regression with tuned LightGBM.

Both models are fitted on the same 100,000-row training sample for a consistent comparison. The ensemble is evaluated on the full validation dataset and compared with the individual models.

In [10]:
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

voting_model = VotingRegressor(
    estimators=[
        ("ridge", Ridge(alpha=1.0, solver="lsqr")),
        ("lightgbm", tuned_lgbm),
    ],
    n_jobs=1
)

voting_model.fit(X_sample, y_sample)

voting_predictions = voting_model.predict(X_valid)

voting_mae = mean_absolute_error(y_valid, voting_predictions)
voting_rmse = np.sqrt(
    mean_squared_error(y_valid, voting_predictions)
)

print(f"Voting MAE (log-price):  {voting_mae:.4f}")
print(f"Voting RMSE (log-price): {voting_rmse:.4f}")

Voting MAE (log-price):  0.4593
Voting RMSE (log-price): 0.5959


In [11]:
print("Ridge model:", type(voting_model.named_estimators_["ridge"]).__name__)
print("LightGBM model:", type(voting_model.named_estimators_["lightgbm"]).__name__)

print(
    "LightGBM parameters match:",
    voting_model.named_estimators_["lightgbm"].get_params()
    == tuned_lgbm.get_params()
)

Ridge model: Ridge
LightGBM model: LGBMRegressor
LightGBM parameters match: True


In [12]:
import pandas as pd

results = pd.DataFrame([
    {
        "Model": "DummyRegressor",
        "MAE": baseline_mae,
        "RMSE": baseline_rmse,
        "Training Data": "Full training set"
    },
    {
        "Model": "Ridge Regression",
        "MAE": ridge_mae,
        "RMSE": ridge_rmse,
        "Training Data": "Full training set"
    },
    {
        "Model": "Random Forest",
        "MAE": rf_mae,
        "RMSE": rf_rmse,
        "Training Data": "100,000-row sample"
    },
    {
        "Model": "Initial LightGBM",
        "MAE": lgbm_mae,
        "RMSE": lgbm_rmse,
        "Training Data": "100,000-row sample"
    },
    {
        "Model": "Tuned LightGBM",
        "MAE": tuned_mae,
        "RMSE": tuned_rmse,
        "Training Data": "100,000-row sample"
    },
    {
        "Model": "Voting Ensemble",
        "MAE": voting_mae,
        "RMSE": voting_rmse,
        "Training Data": "100,000-row sample"
    }
])

results = results.sort_values("MAE").reset_index(drop=True)
display(results.round(4))

results.to_csv("../data/model_comparison.csv", index=False)
print("Results saved to ../data/model_comparison.csv")

,Model,MAE,RMSE,Training Data
0,Tuned LightGBM,0.4109,0.5411,"100,000-row sample"
1,Initial LightGBM,0.4206,0.5519,"100,000-row sample"
2,Random Forest,0.4535,0.5937,"100,000-row sample"
3,Voting Ensemble,0.4593,0.5959,"100,000-row sample"
4,Ridge Regression,0.5643,0.7219,Full training set
5,DummyRegressor,0.5817,0.7455,Full training set


Results saved to ../data/model_comparison.csv


In [13]:
import joblib
from pathlib import Path

preprocessor_path = Path("../data/preprocessor.joblib")
text_pipeline_path = Path("../data/text_pipeline.joblib")
model_path = Path("../data/models/tuned_lightgbm.joblib")

preprocessor = joblib.load(preprocessor_path)
text_pipeline = joblib.load(text_pipeline_path)
saved_model = joblib.load(model_path)

print("Preprocessor type:", type(preprocessor))
print("Preprocessor methods:",
      [m for m in dir(preprocessor) if not m.startswith("_")])

print("\nText pipeline components:", list(text_pipeline.keys()))
print("\nSaved model components:", list(saved_model.keys()))

AttributeError: Can't get attribute 'MercariPreprocessor' on <module '__main__'>

In [14]:
import os
import sys
import joblib
from pathlib import Path

# Make the project's src directory importable
sys.path.insert(0, os.path.abspath("../src"))

# Import the class required to deserialize the saved preprocessor
from preprocessing import MercariPreprocessor

# Load the saved artifacts
preprocessor_path = Path("../data/preprocessor.joblib")
text_pipeline_path = Path("../data/text_pipeline.joblib")
model_path = Path("../data/models/tuned_lightgbm.joblib")

preprocessor = joblib.load(preprocessor_path)
text_pipeline = joblib.load(text_pipeline_path)
saved_model = joblib.load(model_path)

print("Preprocessor type:", type(preprocessor))
print("Text pipeline components:", list(text_pipeline.keys()))
print("Saved model components:", list(saved_model.keys()))

Preprocessor type: <class 'preprocessing.MercariPreprocessor'>
Text pipeline components: ['name_vectorizer', 'description_vectorizer', 'name_svd', 'description_svd']
Saved model components: ['model', 'best_params', 'validation_mae', 'validation_rmse']


In [15]:
import inspect

# Inspect the preprocessing class
print("=== PREPROCESSOR SOURCE ===")
print(inspect.getsource(MercariPreprocessor))

# Inspect the saved feature columns
features = joblib.load("../data/model_features.joblib")
print("\n=== FEATURE COLUMNS ===")
print(features["X_train"].columns.tolist())

# Check the text pipeline output dimensions
print("\n=== TEXT TRANSFORMERS ===")
for name in ["name_svd", "description_svd"]:
    print(name, "components:", text_pipeline[name].n_components)

=== PREPROCESSOR SOURCE ===
class MercariPreprocessor(BaseEstimator, TransformerMixin):
    """Missing values, text cleaning, feature engineering and encoding.

    fit() learns the label encoders from TRAIN data only; transform() applies
    exactly the same steps to any dataframe (train, validation or test).
    """

    def __init__(self):
        self.brand_encoder = LabelEncoder()
        self.category_encoders = {}
        self.is_fitted = False

    # ---------- fit / transform ----------
    def fit(self, X, y=None):
        # Reuse the transform steps so fit and transform can never disagree
        X = self._engineer_features(self._handle_missing_values(X.copy()))

        # 'unknown' is always added so unseen values can fall back to it
        self.brand_encoder.fit(
            pd.concat([X["brand_name"].astype(str), pd.Series([UNKNOWN])])
        )
        for col in ["category_1", "category_2", "category_3"]:
            le = LabelEncoder()
            le.fit(pd.concat([X[

In [16]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))

from prediction_pipeline import MercariPricePredictor

predictor = MercariPricePredictor()

sample_listing = {
    "name": "Nike running shoes",
    "item_condition_id": 2,
    "category_name": "Women/Shoes/Athletic",
    "brand_name": "Nike",
    "item_description": "Lightly used running shoes in good condition",
    "shipping": 0,
}

features = predictor.prepare_features(sample_listing)

print("Feature shape:", features.shape)
print("Feature count:", len(features.columns))
print("Missing values:", features.isna().sum().sum())

predicted_price = predictor.predict(sample_listing)[0]
print(f"Predicted price: ${predicted_price:.2f}")

Feature shape: (1, 212)
Feature count: 212
Missing values: 0
Predicted price: $28.05


In [17]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use 100 validation rows for a quick initial check
n = 100
sample_indices = X_valid.index[:n]

# Load the corresponding processed validation rows
valid_df = pd.read_csv("../data/valid_processed.csv")
valid_sample = valid_df.iloc[:n].copy()

# Reconstruct the raw listing inputs
raw_columns = [
    "name",
    "item_condition_id",
    "category_name",
    "brand_name",
    "item_description",
    "shipping",
]

raw_sample = pd.read_csv(
    "../data/valid_processed.csv",
    usecols=lambda col: col in raw_columns
).iloc[:n]

print("Available columns:", raw_sample.columns.tolist())

Available columns: ['name', 'item_condition_id', 'category_name', 'brand_name', 'shipping', 'item_description']


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Predict prices in the original dollar scale
predicted_prices = predictor.predict(raw_sample)

# Actual validation targets are log-transformed
actual_log_prices = np.asarray(y_valid[:len(raw_sample)])

# Convert predictions back to log-price space for a fair comparison
predicted_log_prices = np.log1p(predicted_prices)

mae = mean_absolute_error(actual_log_prices, predicted_log_prices)
rmse = np.sqrt(mean_squared_error(actual_log_prices, predicted_log_prices))

print(f"Validation rows tested: {len(raw_sample)}")
print(f"Pipeline MAE (log-price): {mae:.4f}")
print(f"Pipeline RMSE (log-price): {rmse:.4f}")
print(f"Previously recorded MAE: {saved_model['validation_mae']:.4f}")
print(f"Previously recorded RMSE: {saved_model['validation_rmse']:.4f}")

Validation rows tested: 100
Pipeline MAE (log-price): 0.3853
Pipeline RMSE (log-price): 0.5033
Previously recorded MAE: 0.4109
Previously recorded RMSE: 0.5411


In [19]:
# Compare pipeline predictions with direct model predictions
X_sample_features = X_valid.iloc[:100]

direct_predictions = predictor.model.predict(X_sample_features)
pipeline_predictions = predictor.predict(raw_sample)

pipeline_log_predictions = np.log1p(pipeline_predictions)

print(
    "Maximum prediction difference:",
    np.max(np.abs(direct_predictions - pipeline_log_predictions))
)

Maximum prediction difference: 0.0


In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Process all validation listings through the pipeline
all_raw_valid = pd.read_csv(
    "../data/valid_processed.csv",
    usecols=[
        "name",
        "item_condition_id",
        "category_name",
        "brand_name",
        "item_description",
        "shipping",
    ],
)

all_predictions = predictor.predict(all_raw_valid)
all_predictions_log = np.log1p(all_predictions)

actual_log = np.asarray(y_valid)

full_mae = mean_absolute_error(actual_log, all_predictions_log)
full_rmse = np.sqrt(
    mean_squared_error(actual_log, all_predictions_log)
)

print(f"Full validation MAE:  {full_mae:.4f}")
print(f"Full validation RMSE: {full_rmse:.4f}")

Full validation MAE:  0.4109
Full validation RMSE: 0.5411
